#### Installing Packages

In [6]:
# 1. Install BiocManager if you don't already have it
if (!require("BiocManager", quietly = TRUE)) {
    install.packages("BiocManager")
}

# 2. Use BiocManager to install ensembldb
BiocManager::install("ensembldb")

# (Optional but likely needed based on your previous code)
# Install AnnotationHub if you haven't already:
BiocManager::install("AnnotationHub")


'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.0 (2026-04-24)

Installing package(s) 'ensembldb'

also installing the dependencies ‘GenomicFeatures’, ‘rtracklayer’


Old packages: 'broom', 'bslib', 'openssl', 'readxl', 'xtable'

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.0 (2026-04-24)

Installing package(s) 'AnnotationHub'

also installing the dependencies ‘filelock’, ‘BiocFileCache’, ‘BiocBaseUtils’


Old packages: 'broom', 'bslib', 'openssl', 'readxl', 'xtable'



In [7]:
# ---------------------------------------------------------
# 1. SETUP & DIRECTORIES
# ---------------------------------------------------------
dir.create("/content/compbioass", recursive = TRUE)
setwd("/content/compbioass") # Example: Using a path within Colab's temporary storage

library(ensembldb)
library(AnnotationHub)

# Create the AnnotationHub cache directory if it doesn't exist
dir.create("/content/compbioass/AH_cache", recursive = TRUE)
setAnnotationHubOption("CACHE", "/content/compbioass/AH_cache") # Example path

ah <- AnnotationHub()
edb <- ah[["AH119325"]]

print("Environment successfully set up!")

Warning message in dir.create("/content/compbioass", recursive = TRUE):
“'/content/compbioass' already exists”
Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Loading required package: GenomicRanges

Loading required package: stats4



downloading 1 resources

retrieving 1 resource



loading from cache



[1] "Environment successfully set up!"


In [8]:
# ---------------------------------------------------------
# 2. DEFINE YOUR GENE
# ---------------------------------------------------------
my_gene <- "BRCA1"


In [9]:
# ---------------------------------------------------------
# 3. FETCH TRANSCRIPTS & PROTEINS
# ---------------------------------------------------------
gene_transcripts <- transcripts(edb, filter = ~ gene_name == my_gene)

# Extracting and viewing the transcript IDs
transcript_ids <- gene_transcripts$tx_id
cat("\n--- List of Transcripts ---\n")
print(transcript_ids)

# Fetching all proteins and extracting for the gene
gene_proteins <- proteins(edb, filter = ~ gene_name == my_gene)
protein_ids <- na.omit(gene_proteins$protein_id)

cat("\n--- List of Proteins ---\n")
print(protein_ids)



--- List of Transcripts ---
 [1] "ENST00000497488" "ENST00000489037" "ENST00000478531" "ENST00000357654"
 [5] "ENST00000473961" "ENST00000477152" "ENST00000352993" "ENST00000493919"
 [9] "ENST00000494123" "ENST00000471181" "ENST00000652672" "ENST00000634433"
[13] "LRG_292t1"       "ENST00000476777" "ENST00000700081" "ENST00000470026"
[17] "ENST00000713676" "ENST00000618469" "ENST00000461574" "ENST00000644555"
[21] "ENST00000468300" "ENST00000700082" "ENST00000644379" "ENST00000484087"
[25] "ENST00000586385" "ENST00000591534" "ENST00000591849" "ENST00000493795"
[29] "ENST00000461221" "ENST00000491747" "ENST00000472490" "ENST00000700182"
[33] "ENST00000621897" "ENST00000354071" "ENST00000700183" "ENST00000492859"
[37] "ENST00000642945" "ENST00000700184" "ENST00000461798" "ENST00000700083"
[41] "ENST00000700185" "ENST00000700186"

--- List of Proteins ---
 [1] "ENSP00000418986" "ENSP00000420781" "ENSP00000420412" "ENSP00000350283"
 [5] "ENSP00000420201" "ENSP00000419988" "ENSP00000312236

In [10]:
# ---------------------------------------------------------
# 4. MAPPING LOOP
# ---------------------------------------------------------
mapping_results <- list()

for (pid in protein_ids) {

  # Get the protein sequence length to know the start and end of the amino acid chain
  prot_info <- proteins(edb, filter = ~ protein_id == pid)
  prot_length <- nchar(prot_info$protein_sequence)

  # Create an IRanges object representing the full amino acid sequence
  aa_range <- IRanges(start = 1, end = prot_length)
  names(aa_range) <- pid

  # Map the amino acid coordinates to the genome
  mapped_coords <- proteinToGenome(aa_range, edb)

  # Store the result as a data frame
  mapping_results[[pid]] <- as.data.frame(mapped_coords[[1]])
}


Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ... 
1 found

Checking CDS and protein sequence lengths ... 
1/1 OK

Fetching CDS for 1 proteins ..

In [11]:
# ---------------------------------------------------------
# 5. COMBINE AND SAVE
# ---------------------------------------------------------
# Combine all individual protein mappings into one big, clean table
final_map_df <- do.call(rbind, mapping_results)

# Save this map to a TSV file
output_filename <- paste0(my_gene, "_protein_to_genome_map.tsv")

write.table(final_map_df, file = output_filename,
            sep = "\t", row.names = FALSE, quote = FALSE)

cat("\n--- Success! ---\n")
cat("Mapping complete! File saved as:", output_filename, "in your working directory.\n")


--- Success! ---
Mapping complete! File saved as: BRCA1_protein_to_genome_map.tsv in your working directory.
